# BÀI THỰC HÀNH TUẦN 6 (Phần 1): CÁC KỸ THUẬT PHÂN LOẠI
## ÁP DỤNG DECISION TREE, SVM, NEURAL NETWORK & BAYESIAN CLASSIFICATION

> **Dữ liệu:** Bank Marketing Dataset (UCI) — `bank-full.csv`  
> **Bài toán:** Phân loại nhị phân — Dự đoán khách hàng có đăng ký tiền gửi kỳ hạn hay không (`y`: yes/no)  
> **Tiếp nối:** Sử dụng cùng bộ dữ liệu đã phân tích từ Tuần 1–3 để đảm bảo sự nhất quán  

---

### Tại sao chọn Bank Marketing cho bài toán phân loại?

| Tiêu chí | Phù hợp? | Giải thích |
|-----------|----------|------------|
| Biến mục tiêu nhị phân | ✅ | `y` = yes/no → Bài toán phân loại 2 lớp chuẩn |
| Dữ liệu đủ lớn | ✅ | 45,211 mẫu → đủ cho train/test và tránh overfitting |
| Hỗn hợp đặc trưng | ✅ | Vừa có biến số (age, balance) vừa có biến phân loại (job, marital) |
| Mất cân bằng lớp | ✅ | 88.3% "no" vs 11.7% "yes" → Thực tế, phải xử lý class imbalance |

---

## 1. THIẾT LẬP MÔI TRƯỜNG & LOAD DỮ LIỆU

### 1.1. Import thư viện

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
np.seterr(all='ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

# Tiền xử lý
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV

# Mô hình phân loại
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB, CategoricalNB

# Đánh giá
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 100})
np.random.seed(42)

print("Đã nạp xong thư viện.")

### 1.2. Load & khám phá dữ liệu

In [ ]:
df = pd.read_csv('../data/bank-full.csv', sep=';')
print(f"Kích thước dữ liệu: {df.shape[0]:,} dòng x {df.shape[1]} cột")
print(f"\nPhân bố biến mục tiêu 'y':")
display(df['y'].value_counts().to_frame('Số lượng').assign(**{'Tỷ lệ (%)': lambda x: x['Số lượng'] / len(df) * 100}))
df.head()

---
## 2. TIỀN XỬ LÝ DỮ LIỆU

### 2.1. Mã hóa biến phân loại & chuẩn bị đặc trưng

**Chiến lược mã hóa:**
- Biến nhị phân (`default`, `housing`, `loan`): Label Encoding (0/1)
- Biến danh mục nhiều giá trị (`job`, `marital`, `education`, `contact`, `month`, `poutcome`): One-Hot Encoding
- Biến mục tiêu `y`: 1 = yes, 0 = no

In [ ]:
df_processed = df.copy()

# Mã hóa biến mục tiêu
df_processed['y'] = (df_processed['y'] == 'yes').astype(int)

# Mã hóa biến nhị phân
for col in ['default', 'housing', 'loan']:
    df_processed[col] = (df_processed[col] == 'yes').astype(int)

# One-Hot Encoding cho biến danh mục
cat_cols = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']
df_processed = pd.get_dummies(df_processed, columns=cat_cols, drop_first=True)

print(f"Kích thước sau mã hóa: {df_processed.shape[0]:,} dòng x {df_processed.shape[1]} cột")
print(f"Số đặc trưng: {df_processed.shape[1] - 1}")

### 2.2. Chia tập train/test & chuẩn hóa

In [ ]:
X = df_processed.drop('y', axis=1)
y = df_processed['y']

# Chia dữ liệu: 80% train, 20% test (stratified để giữ tỷ lệ lớp)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Chuẩn hóa StandardScaler — Rất quan trọng cho SVM và Neural Network
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]:,} mẫu | Test: {X_test.shape[0]:,} mẫu")
print(f"Tỷ lệ lớp 'yes' trong train: {y_train.mean():.2%}")
print(f"Tỷ lệ lớp 'yes' trong test:  {y_test.mean():.2%}")

---
## 3. MÔ HÌNH 1: DECISION TREE (CÂY QUYẾT ĐỊNH)

### Tại sao Decision Tree phù hợp với Bank Marketing?

| Ưu điểm | Giải thích |
|----------|------------|
| **Dễ diễn giải** | Ngân hàng cần giải thích cho quản lý tại sao một khách hàng được dự đoán "yes" → Cây quyết định trực quan nhất |
| **Xử lý tốt hỗn hợp biến** | Không cần chuẩn hóa, xử lý được cả biến liên tục lẫn phân loại |
| **Phát hiện tương tác phi tuyến** | Tự động phát hiện các quy tắc phân nhánh phức tạp (e.g. "tuổi > 50 VÀ balance > 1000") |

⚠️ **Nhược điểm cần lưu ý:** Dễ overfitting nếu không giới hạn độ sâu → Dùng pruning và cross-validation.

### 3.1. Huấn luyện Decision Tree

In [ ]:
# Decision Tree không cần dữ liệu chuẩn hóa → dùng X_train gốc
dt_model = DecisionTreeClassifier(
    max_depth=8,           # Giới hạn độ sâu để tránh overfitting
    min_samples_split=50,  # Mỗi nút phải có ít nhất 50 mẫu mới tiếp tục chia
    min_samples_leaf=20,   # Mỗi lá phải có ít nhất 20 mẫu
    class_weight='balanced',  # Xử lý mất cân bằng lớp (88% no vs 12% yes)
    random_state=42
)

start = time.time()
dt_model.fit(X_train, y_train)
dt_train_time = time.time() - start

y_pred_dt = dt_model.predict(X_test)
y_prob_dt = dt_model.predict_proba(X_test)[:, 1]

print(f"Thời gian huấn luyện: {dt_train_time:.3f}s")
print(f"Số nút lá: {dt_model.get_n_leaves()}")
print(f"Độ sâu thực tế: {dt_model.get_depth()}")
print(f"\n{classification_report(y_test, y_pred_dt, target_names=['No', 'Yes'])}")

### 3.2. Trực quan hóa cây quyết định (top 4 tầng)

In [ ]:
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(
    dt_model, max_depth=3, feature_names=X.columns,
    class_names=['No', 'Yes'], filled=True, rounded=True,
    fontsize=8, ax=ax, proportion=True
)
ax.set_title('Decision Tree — Bank Marketing (top 3 tầng)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3. Tầm quan trọng đặc trưng (Feature Importance)

In [ ]:
feat_imp = pd.Series(dt_model.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.head(15).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Feature Importance')
ax.set_title('Top 15 đặc trưng quan trọng nhất — Decision Tree', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 5 đặc trưng quan trọng nhất:")
for i, (feat, imp) in enumerate(feat_imp.head(5).items(), 1):
    print(f"  {i}. {feat}: {imp:.4f}")

---
## 4. MÔ HÌNH 2: SUPPORT VECTOR MACHINE (SVM)

### Phân tích tính phù hợp của SVM với Bank Marketing

| Tiêu chí | Đánh giá |
|----------|----------|
| **Hiệu quả với dữ liệu nhiều chiều** | ✅ Sau One-Hot Encoding có ~40+ đặc trưng → SVM mạnh ở không gian nhiều chiều |
| **Khả năng tổng quát hóa tốt** | ✅ Tìm siêu phẳng margin tối đa → ít overfitting hơn Decision Tree |
| **Kernel trick** | ✅ Kernel RBF bắt được quan hệ phi tuyến giữa các đặc trưng |

⚠️ **Nhược điểm:**
- **Chậm với dữ liệu lớn**: Độ phức tạp O(n²) ~ O(n³) → Với 45K mẫu, SVM sẽ khá chậm
- **Yêu cầu chuẩn hóa**: Phải dùng StandardScaler (đã chuẩn bị ở bước 2.2)
- **Khó diễn giải**: Không trực quan như Decision Tree

→ Để tiết kiệm thời gian, ta sẽ dùng **mẫu con ngẫu nhiên 10,000 mẫu** cho huấn luyện SVM.

### 4.1. Huấn luyện SVM

In [ ]:
# Lấy mẫu con 10,000 mẫu để SVM chạy trong thời gian hợp lý
# (SVM có độ phức tạp O(n^2) ~ O(n^3), 45K mẫu sẽ rất chậm)
sample_size = 10000
np.random.seed(42)
sample_idx = np.random.choice(len(X_train_scaled), size=sample_size, replace=False)
X_train_svm = X_train_scaled[sample_idx]
y_train_svm = y_train.iloc[sample_idx]

print(f"Số mẫu dùng cho SVM: {sample_size:,} / {len(X_train_scaled):,}")
print(f"Tỷ lệ lớp 'yes' trong mẫu SVM: {y_train_svm.mean():.2%}")

# SVM với kernel RBF — dùng dữ liệu ĐÃ chuẩn hóa
svm_model = SVC(
    kernel='rbf',          # Kernel phi tuyến
    C=1.0,                 # Regularization parameter
    gamma='scale',         # γ = 1 / (n_features * var(X))
    class_weight='balanced',  # Xử lý mất cân bằng lớp
    probability=True,      # Cho phép predict_proba (dùng Platt scaling)
    random_state=42
)

start = time.time()
svm_model.fit(X_train_svm, y_train_svm)
svm_train_time = time.time() - start

y_pred_svm = svm_model.predict(X_test_scaled)
y_prob_svm = svm_model.predict_proba(X_test_scaled)[:, 1]

print(f"\nThời gian huấn luyện: {svm_train_time:.1f}s")
print(f"Số support vectors: {svm_model.n_support_}")
print(f"\n{classification_report(y_test, y_pred_svm, target_names=['No', 'Yes'])}")

---
## 5. MÔ HÌNH 3: NEURAL NETWORK (MẠNG NƠ-RON)

### Phân tích tính phù hợp của Neural Network

| Tiêu chí | Đánh giá |
|----------|----------|
| **Bắt được quan hệ phi tuyến phức tạp** | ✅ Nhiều lớp ẩn + hàm kích hoạt phi tuyến |
| **Tự động học đặc trưng** | ✅ Không cần thiết kế đặc trưng thủ công |
| **Mạnh với dữ liệu lớn** | ✅ 45K mẫu là vừa đủ cho MLP đơn giản |

⚠️ **Nhược điểm:**
- **Hộp đen**: Rất khó giải thích tại sao mô hình đưa ra dự đoán → Không lý tưởng cho ngân hàng cần giải trình
- **Yêu cầu chuẩn hóa**: Bắt buộc phải StandardScaler
- **Nhạy cảm với hyperparameter**: Số lớp, số neuron, learning rate... cần điều chỉnh cẩn thận

### 5.1. Huấn luyện Neural Network (MLP)

In [ ]:
# Multi-Layer Perceptron — dùng dữ liệu ĐÃ chuẩn hóa
nn_model = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),  # 3 lớp ẩn: 128 → 64 → 32 neuron
    activation='relu',                 # Hàm kích hoạt ReLU
    solver='adam',                     # Optimizer Adam
    alpha=0.001,                       # L2 regularization
    batch_size=256,                    # Mini-batch size
    learning_rate='adaptive',          # Giảm learning rate khi loss không giảm
    learning_rate_init=0.001,
    max_iter=300,                      # Tối đa 300 epochs
    early_stopping=True,               # Dừng sớm khi validation loss tăng
    validation_fraction=0.1,           # 10% train dùng cho validation
    random_state=42
)

start = time.time()
nn_model.fit(X_train_scaled, y_train)
nn_train_time = time.time() - start

y_pred_nn = nn_model.predict(X_test_scaled)
y_prob_nn = nn_model.predict_proba(X_test_scaled)[:, 1]

print(f"Thời gian huấn luyện: {nn_train_time:.1f}s")
print(f"Số epochs thực tế: {nn_model.n_iter_}")
print(f"Loss cuối cùng: {nn_model.loss_:.4f}")
print(f"\n{classification_report(y_test, y_pred_nn, target_names=['No', 'Yes'])}")

### 5.2. Quá trình hội tụ của Neural Network (Loss curve)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(nn_model.loss_curve_, label='Training Loss', color='steelblue', linewidth=2)
if hasattr(nn_model, 'validation_scores_') and nn_model.validation_scores_ is not None:
    ax2 = ax.twinx()
    ax2.plot(nn_model.validation_scores_, label='Validation Accuracy', color='coral', linewidth=2, linestyle='--')
    ax2.set_ylabel('Validation Accuracy')
    ax2.legend(loc='center right')
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Quá trình huấn luyện Neural Network (MLP)', fontweight='bold')
ax.legend(loc='center left')
plt.tight_layout()
plt.show()

---
## 6. MÔ HÌNH 4: BAYESIAN CLASSIFICATION (NAIVE BAYES)

### Phân tích tính phù hợp & KHÔNG phù hợp

#### ✅ Điểm phù hợp:
| Tiêu chí | Giải thích |
|----------|------------|
| **Nhanh chóng** | Tính toán xác suất có điều kiện rất nhanh, phù hợp triển khai real-time |
| **Hoạt động tốt với ít dữ liệu** | Ngay cả khi dữ liệu ít, NB vẫn cho kết quả hợp lý |
| **Có thể diễn giải** | Xác suất posterior cho biết "niềm tin" của mô hình |

#### ⚠️ Điểm KHÔNG phù hợp (Quan trọng):
| Vấn đề | Giải thích |
|--------|------------|
| **Giả định "Naive" (độc lập có điều kiện)** | NB giả định tất cả đặc trưng độc lập nhau — **Điều này SAI** với Bank Marketing vì các biến rõ ràng có tương quan (e.g. `age` & `balance`, `job` & `education`, `housing` & `loan`) |
| **GaussianNB giả định phân phối chuẩn** | Nhiều biến số trong dataset không tuân theo phân phối chuẩn (e.g. `balance` lệch phải rất mạnh, `duration` lệch phải) |
| **Không bắt được tương tác giữa các biến** | Khác với Decision Tree hay NN, NB không thể mô hình hóa "tuổi > 50 VÀ balance > 1000" |

→ **Kết luận:** Naive Bayes vẫn được áp dụng như một **baseline đơn giản** để so sánh, nhưng kỳ vọng hiệu suất sẽ thấp hơn các mô hình khác do vi phạm giả định độc lập.

### 6.1. Huấn luyện Gaussian Naive Bayes

In [ ]:
# Kiểm tra giả định phân phối chuẩn cho các biến số chính
num_cols = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']
print("=== Kiểm tra giả định phân phối chuẩn (Skewness & Kurtosis) ===")
print(f"{'Biến':<12} {'Skewness':>10} {'Kurtosis':>10}  Nhận xét")
print("-" * 60)
for col in num_cols:
    skew = df[col].skew()
    kurt = df[col].kurtosis()
    note = "⚠️ Lệch mạnh" if abs(skew) > 1 else ("~ Chấp nhận" if abs(skew) < 0.5 else "Lệch nhẹ")
    print(f"{col:<12} {skew:>10.2f} {kurt:>10.2f}  {note}")

print("\n→ Nhiều biến vi phạm giả định phân phối chuẩn của GaussianNB.")
print("  NB vẫn được áp dụng như baseline, nhưng kỳ vọng hiệu suất thấp hơn.")

In [ ]:
# Gaussian Naive Bayes — dùng dữ liệu ĐÃ chuẩn hóa
nb_model = GaussianNB()

start = time.time()
nb_model.fit(X_train_scaled, y_train)
nb_train_time = time.time() - start

y_pred_nb = nb_model.predict(X_test_scaled)
y_prob_nb = nb_model.predict_proba(X_test_scaled)[:, 1]

print(f"Thời gian huấn luyện: {nb_train_time:.4f}s (Nhanh nhất trong 4 mô hình!)")
print(f"\n{classification_report(y_test, y_pred_nb, target_names=['No', 'Yes'])}")

### 6.2. Kiểm chứng: Tương quan giữa các đặc trưng (vi phạm giả định Naive)

In [ ]:
# Hiển thị ma trận tương quan giữa các biến số chính
fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = df[num_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Ma trận tương quan — Kiểm chứng giả định "Naive"', fontweight='bold')
plt.tight_layout()
plt.show()

print("→ Các cặp biến có tương quan đáng kể:")
for i in range(len(num_cols)):
    for j in range(i+1, len(num_cols)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.1:
            print(f"  • {num_cols[i]} — {num_cols[j]}: r = {r:.3f}")
print("→ Xác nhận vi phạm giả định độc lập của Naive Bayes.")

---
## 7. SO SÁNH TỔNG HỢP 4 MÔ HÌNH

### 7.1. Bảng so sánh các chỉ số đánh giá

In [ ]:
models = {
    'Decision Tree': (y_pred_dt, y_prob_dt, dt_train_time),
    'SVM (RBF)':     (y_pred_svm, y_prob_svm, svm_train_time),
    'Neural Network': (y_pred_nn, y_prob_nn, nn_train_time),
    'Naive Bayes':   (y_pred_nb, y_prob_nb, nb_train_time),
}

results = []
for name, (y_pred, y_prob, train_time) in models.items():
    results.append({
        'Mô hình': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision (Yes)': precision_score(y_test, y_pred),
        'Recall (Yes)': recall_score(y_test, y_pred),
        'F1 (Yes)': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_prob),
        'Thời gian (s)': train_time
    })

df_results = pd.DataFrame(results).set_index('Mô hình')

# Highlight giá trị tốt nhất mỗi cột
def highlight_max(s):
    if s.name == 'Thời gian (s)':
        return ['font-weight: bold' if v == s.min() else '' for v in s]
    return ['font-weight: bold' if v == s.max() else '' for v in s]

display(df_results.style.apply(highlight_max).format({
    'Accuracy': '{:.4f}', 'Precision (Yes)': '{:.4f}',
    'Recall (Yes)': '{:.4f}', 'F1 (Yes)': '{:.4f}',
    'AUC-ROC': '{:.4f}', 'Thời gian (s)': '{:.3f}'
}))

### 7.2. Confusion Matrix so sánh

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (name, (y_pred, y_prob, _)) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['No', 'Yes']).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(name, fontweight='bold', fontsize=11)

plt.suptitle('Confusion Matrix — So sánh 4 mô hình phân loại', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 7.3. ROC Curve so sánh

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']

# ROC Curve
ax = axes[0]
for (name, (_, y_prob, _)), color in zip(models.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)

# Precision-Recall Curve
ax = axes[1]
for (name, (_, y_prob, _)), color in zip(models.items(), colors):
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    ax.plot(recall, precision, label=name, color=color, linewidth=2)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve', fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('So sánh hiệu suất 4 mô hình phân loại — Bank Marketing', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 7.4. So sánh thời gian huấn luyện

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
train_times = pd.Series({
    'Decision Tree': dt_train_time,
    'SVM (RBF)': svm_train_time,
    'Neural Network': nn_train_time,
    'Naive Bayes': nb_train_time
})
bars = train_times.plot(kind='barh', ax=ax, color=colors)
for i, (v, name) in enumerate(zip(train_times, train_times.index)):
    ax.text(v + 0.01, i, f'{v:.3f}s', va='center', fontweight='bold')
ax.set_xlabel('Thời gian (giây)')
ax.set_title('Thời gian huấn luyện các mô hình', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. KẾT LUẬN & ĐÁNH GIÁ TỔNG QUAN

### 8.1. Tổng hợp nhận xét

In [ ]:
print("="*80)
print("TỔNG KẾT — SO SÁNH 4 KỸ THUẬT PHÂN LOẠI TRÊN BANK MARKETING DATASET")
print("="*80)

# Tìm mô hình tốt nhất theo từng tiêu chí
best_acc = df_results['Accuracy'].idxmax()
best_f1 = df_results['F1 (Yes)'].idxmax()
best_auc = df_results['AUC-ROC'].idxmax()
best_recall = df_results['Recall (Yes)'].idxmax()
fastest = df_results['Thời gian (s)'].idxmin()

print(f"\n📊 Mô hình tốt nhất theo:")
print(f"   • Accuracy:        {best_acc} ({df_results.loc[best_acc, 'Accuracy']:.4f})")
print(f"   • F1-Score (Yes):  {best_f1} ({df_results.loc[best_f1, 'F1 (Yes)']:.4f})")
print(f"   • AUC-ROC:         {best_auc} ({df_results.loc[best_auc, 'AUC-ROC']:.4f})")
print(f"   • Recall (Yes):    {best_recall} ({df_results.loc[best_recall, 'Recall (Yes)']:.4f})")
print(f"   • Nhanh nhất:      {fastest} ({df_results.loc[fastest, 'Thời gian (s)']:.4f}s)")

print(f"\n" + "="*80)
print("PHÂN TÍCH CHI TIẾT TỪNG MÔ HÌNH")
print("="*80)

analysis = [
    ("1. Decision Tree", [
        "✅ Dễ diễn giải nhất — phù hợp cho ngân hàng cần giải trình quyết định",
        "✅ Nhanh, không cần chuẩn hóa dữ liệu",
        "✅ Cung cấp feature importance trực quan",
        "⚠️ Dễ overfitting nếu không pruning cẩn thận",
    ]),
    ("2. SVM (RBF Kernel)", [
        "✅ Tổng quát hóa tốt nhờ margin maximization",
        "✅ Mạnh với dữ liệu nhiều chiều",
        "⚠️ RẤT CHẬM với dữ liệu lớn — phải dùng mẫu con 10K/36K",
        "⚠️ Khó diễn giải — không phù hợp cho bối cảnh ngân hàng cần giải trình",
    ]),
    ("3. Neural Network (MLP)", [
        "✅ Bắt được quan hệ phi tuyến phức tạp nhất",
        "✅ Có thể mở rộng cho dữ liệu rất lớn",
        "⚠️ Hộp đen hoàn toàn — không giải thích được",
        "⚠️ Nhạy cảm với hyperparameter, cần nhiều thí nghiệm",
    ]),
    ("4. Naive Bayes (Gaussian)", [
        "✅ Nhanh nhất trong tất cả — phù hợp real-time",
        "✅ Đơn giản, dễ triển khai, ít hyperparameter",
        "⚠️ Vi phạm giả định 'naive' (các đặc trưng KHÔNG độc lập)",
        "⚠️ Nhiều biến không tuân theo phân phối chuẩn (balance, duration bị lệch)",
        "⚠️ Kỳ vọng hiệu suất thấp nhất do vi phạm các giả định cốt lõi",
    ]),
]

for title, points in analysis:
    print(f"\n{title}")
    for p in points:
        print(f"   {p}")

print(f"\n" + "="*80)
print("KHUYẾN NGHỊ CHO BÀI TOÁN BANK MARKETING")
print("="*80)
print("""
• Nếu cần DỄ DIỄN GIẢI (giải trình cho quản lý): → Decision Tree
• Nếu cần HIỆU SUẤT CAO NHẤT:                   → Neural Network hoặc SVM
• Nếu cần BASELINE NHANH để so sánh:              → Naive Bayes
• Nếu muốn CÂN BẰNG giữa hiệu suất & diễn giải: → Ensemble (Phần 2 — Tuần 6)
""")